In [1]:
"""
MDCG Document Parsing Pipeline
===============================
Parses MDCG guidance PDFs into clean, structured markdown using
LlamaParse v2 (agentic tier) + Python post-processing.

Outputs:
  - Clean markdown with consolidated footnotes and correct heading hierarchy
  - Structured YAML decision flowcharts (Charts A–E)
"""
import os
import re
import json
import yaml
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Load .env from project root
_env_path = Path.cwd().parents[1] / ".env" if Path.cwd().name == "sandbox" else Path.cwd() / ".env"
load_dotenv(_env_path, override=False)

# API key — set LLAMA_CLOUD_API_KEY in .env or paste below
LLAMA_API_KEY = os.environ.get("LLAMA_CLOUD_API_KEY", "")
if not LLAMA_API_KEY:
    # ⚠ Paste your key here if not using .env, then add to .env for persistence:
    #   echo 'LLAMA_CLOUD_API_KEY=llx-...' >> /path/to/crss/.env
    LLAMA_API_KEY = ""  # <-- paste key here if needed

if not LLAMA_API_KEY:
    raise RuntimeError(
        "Set LLAMA_CLOUD_API_KEY in your .env or paste it in cell 1.\n"
        "Get one at https://cloud.llamaindex.ai/"
    )

SANDBOX = Path.cwd() if Path.cwd().name == "sandbox" else Path("ingestion/sandbox")
print(f"API key loaded: {LLAMA_API_KEY[:8]}…")
print(f"Sandbox dir:    {SANDBOX}")

API key loaded: llx-XNpF…
Sandbox dir:    /Users/diegobarra/Developer/crss/ingestion/sandbox


In [2]:
# ── MDCG-specific custom prompt for LlamaParse agentic tier ──────────────────

MDCG_CUSTOM_PROMPT = """
You are parsing an official MDCG (Medical Device Coordination Group) guidance
document published by the European Commission. These are regulatory guidance
PDFs with numbered sections, footnotes, and decision-tree flowcharts.

CRITICAL INSTRUCTIONS:

1. CONTINUOUS DOCUMENT — Treat the entire PDF as ONE continuous document.
   Do NOT restart headings or numbering at page boundaries.

2. REMOVE REPEATED HEADERS/FOOTERS — The following text appears on nearly
   every page as a running header or footer. REMOVE every occurrence EXCEPT
   the very first one at the start of the document:
   - "Medical Devices"
   - "Medical Device Coordination Group Document"
   - "MDCG 2020-3 Rev.1" (or any MDCG document reference that repeats)
   Remove page numbers as well.

3. SECTION HIERARCHY — Preserve the original numbered section structure
   exactly. Map section numbers to markdown heading levels:
   - Document title → # (level 1)
   - Top sections (1, 2, 3, 4) → ## (level 2)
   - Subsections (4.1, 4.2, 4.3) → ### (level 3)
   - Sub-subsections (4.3.1, 4.3.2) → #### (level 4)
   - Deeper (4.3.2.1, 4.3.2.2, 4.3.2.3) → ##### (level 5)

4. FOOTNOTES — Collect ALL footnotes from the entire document.
   Place them in a SINGLE section titled "## Footnotes" at the very end
   of the document. Preserve the original footnote numbering (superscript
   numbers become plain numbers like "1.", "2.", etc.). Do NOT scatter
   footnotes at the end of each page — consolidate them ALL at the end.

5. FLOWCHARTS (DECISION TREES) — The document contains decision-tree
   flowcharts labeled "Main Chart", "Chart A", "Chart B", "Chart C",
   "Chart D", "Chart E". For each flowchart:
   - Start with the chart title as a heading
   - Represent each decision node as a numbered step with the question
   - Show Yes/No branches using indented bullet points
   - Use → to indicate flow direction
   - Example format:
     **Step B1**: Change of built-in control mechanism, operating principles,
     source of energy or alarm systems?
       - **Yes** → The change is considered significant
       - **No** → Go to Step B2

6. FORMATTING — Preserve bold (**text**) and italic (*text*) formatting.
   Keep bullet lists as markdown lists. Keep all legal references exactly
   as written (Article numbers, Regulation references, Directive references).

7. EXAMPLES — When the document provides lists of "Non-significant" and
   "Significant" change examples, preserve them as bullet lists under clear
   subheadings.

8. TABLES — If the document contains tables, render them as markdown tables.

9. CONTENT PRESERVATION — Do NOT omit, summarize, or paraphrase any content.
   Include every sentence, every example, every footnote from the original.
"""

print(f"Custom prompt: {len(MDCG_CUSTOM_PROMPT)} chars")

Custom prompt: 2733 chars


In [3]:
# ── Parse PDF with LlamaParse v2 (agentic tier) ────────────────────────────

from llama_cloud import AsyncLlamaCloud

PDF_PATH = SANDBOX / "mdcg_2020_3_rev1.pdf"
assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

client = AsyncLlamaCloud(api_key=LLAMA_API_KEY)

# Upload file
print(f"Uploading {PDF_PATH.name}…")
file_obj = await client.files.create(file=str(PDF_PATH), purpose="parse")
print(f"File ID: {file_obj.id}")

# Parse with agentic tier + custom prompt
print("Parsing with agentic tier (this may take 1-3 minutes)…")
result = await client.parsing.parse(
    file_id=file_obj.id,
    tier="agentic",
    version="latest",
    agentic_options={
        "custom_prompt": MDCG_CUSTOM_PROMPT,
    },
    # Crop top/bottom margins to reduce header/footer repetition
    crop_box={
        "top": 0.07,
        "bottom": 0.04,
        "left": 0.0,
        "right": 0.0,
    },
    output_options={
        "markdown": {
            "annotate_links": True,
        },
    },
    expand=["markdown", "text"],
)

print(f"Parse complete! Pages: {len(result.markdown.pages)}")
print(f"Total markdown chars: {sum(len(p.markdown) for p in result.markdown.pages)}")

Uploading mdcg_2020_3_rev1.pdf…
File ID: e7f0790d-5036-44e9-a89b-1d5280970bcc
Parsing with agentic tier (this may take 1-3 minutes)…
Parse complete! Pages: 21
Total markdown chars: 40736


In [4]:
# ── Preview raw per-page output ─────────────────────────────────────────────

for i, page in enumerate(result.markdown.pages):
    preview = page.markdown[:200].replace("\n", "\\n")
    print(f"Page {i+1} ({len(page.markdown)} chars): {preview}…\n")

Page 1 (831 chars): Medical Device Coordination Group Document MDCG 2020-3 Rev.1\n\n# MDCG 2020-3 Rev.1\n\n# Guidance on significant changes regarding the transitional provision under Article 120 of the MDR with regard to de…

Page 2 (3273 chars): # MDCG 2020-3 revision 1 changes\n\nAdjustments all over the document to align it to Regulation (EU) 2023/607 and guidance MDCG 2022-2\n\n## 1 Introduction\n\nArticle 120(3) of the Medical Device Regulation…

Page 3 (3972 chars): It is essential for legacy devices that their certificates remain valid<sup>7</sup> following changes that are not significant with regard to design or intended purpose and that the required appropria…

Page 4 (2320 chars): (b) MDR, the implementation of such a change would prevent the manufacturer from placing the device on the market under the AIMDD/MDD in accordance with that provision.<sup>13</sup>\n\n## 4 Assessment w…

Page 5 (3377 chars): procedures<sup>14</sup>), changes and their implementation are to be ver

In [5]:
# ── Save raw output for reference ───────────────────────────────────────────

raw_md = "\n\n".join(p.markdown for p in result.markdown.pages)

raw_path = SANDBOX / "mdcg_2020_3_rev1_raw.md"
raw_path.write_text(raw_md, encoding="utf-8")
print(f"Raw markdown saved: {raw_path} ({len(raw_md)} chars)")

Raw markdown saved: /Users/diegobarra/Developer/crss/ingestion/sandbox/mdcg_2020_3_rev1_raw.md (40776 chars)


In [6]:
# ── Post-processing: clean_mdcg_markdown() ─────────────────────────────────

def clean_mdcg_markdown(raw: str) -> tuple[str, dict]:
    """
    Clean LlamaParse output for MDCG documents.
    Returns (cleaned_markdown, metrics_dict).
    """
    metrics = {}
    text = raw

    # ── 1. Remove page-break separator artifacts ───────────────────────────
    # LlamaParse often inserts --- or ---- between pages
    before = len(re.findall(r"^\s*-{3,}\s*$", text, re.MULTILINE))
    text = re.sub(r"\n\s*-{3,}\s*\n", "\n\n", text)
    metrics["separators_removed"] = before

    # ── 2. Remove duplicate headers ────────────────────────────────────────
    # These patterns repeat on every page as running headers/footers.
    # Keep only the FIRST occurrence of the document title block.
    header_patterns = [
        # Exact repeated header lines (as H1/H2/H3)
        r"^#{1,3}\s*Medical Devices\s*$",
        r"^#{1,3}\s*Medical Device Coordination Group Document\s*(?:MDCG\s+\d{4}-\d+\s*(?:Rev\.\d+)?)?\s*$",
        # Plain text versions (no # prefix)
        r"^Medical Devices\s*$",
    ]

    total_header_removals = 0
    for pat in header_patterns:
        matches = list(re.finditer(pat, text, re.MULTILINE))
        if len(matches) > 1:
            # Keep the first, remove the rest (in reverse to preserve offsets)
            for m in reversed(matches[1:]):
                text = text[:m.start()] + text[m.end():]
                total_header_removals += 1
    metrics["duplicate_headers_removed"] = total_header_removals

    # ── 3. Consolidate footnotes ───────────────────────────────────────────
    # Find all footnote sections (## Footnotes or variations)
    footnote_pattern = r"^#{1,3}\s*Footnotes?\s*\n((?:(?!^#{1,3}\s).*\n?)*)"
    footnote_sections = re.findall(footnote_pattern, text, re.MULTILINE)

    # Extract individual footnotes from all sections
    all_footnotes = []
    seen_footnotes = set()
    for section in footnote_sections:
        # Match numbered footnotes: "1. text" or "1 text" or "- text"
        fns = re.findall(
            r"^\s*(?:(\d+)[\.\):]?\s+(.+?))\s*$",
            section, re.MULTILINE
        )
        for num, text_content in fns:
            # Deduplicate by content (normalized)
            norm = re.sub(r"\s+", " ", text_content.strip().lower())
            if norm not in seen_footnotes and len(norm) > 5:
                seen_footnotes.add(norm)
                all_footnotes.append((num, text_content.strip()))

    # Remove all scattered footnote sections from body
    text = re.sub(
        r"\n*^#{1,3}\s*Footnotes?\s*\n(?:(?!^#{1,3}\s).*\n?)*",
        "\n", text, flags=re.MULTILINE
    )
    metrics["footnote_sections_consolidated"] = len(footnote_sections)
    metrics["unique_footnotes"] = len(all_footnotes)

    # ── 4. Normalize heading hierarchy ─────────────────────────────────────
    # Map section numbers to correct heading levels.
    # e.g., "### 4.3.2.1 text" → "##### 4.3.2.1 text" (5 parts = level 5)
    def fix_heading(m):
        hashes = m.group(1)
        num = m.group(2)
        title = m.group(3)
        depth = len(num.split("."))
        # depth 1 (e.g., "1") → ## (level 2), depth 2 → ### (level 3), etc.
        target_level = min(depth + 1, 6)
        return f"{'#' * target_level} {num} {title}"

    text = re.sub(
        r"^(#{1,6})\s+(\d+(?:\.\d+)*)\s+(.+)$",
        fix_heading, text, flags=re.MULTILINE
    )

    # ── 5. Clean whitespace ────────────────────────────────────────────────
    # Collapse 3+ consecutive blank lines to 2
    text = re.sub(r"\n{4,}", "\n\n\n", text)
    # Remove trailing whitespace on lines
    text = re.sub(r"[ \t]+$", "", text, flags=re.MULTILINE)
    # Ensure single newline at end
    text = text.strip() + "\n"

    # ── 6. Append consolidated footnotes ───────────────────────────────────
    if all_footnotes:
        footnotes_block = "\n\n## Footnotes\n\n"
        for num, content in all_footnotes:
            footnotes_block += f"{num}. {content}\n\n"
        text += footnotes_block.rstrip() + "\n"

    return text, metrics


# Run the cleaner
cleaned, metrics = clean_mdcg_markdown(raw_md)

print("Cleaning metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print(f"\nCleaned length: {len(cleaned)} chars (was {len(raw_md)}, "
      f"removed {len(raw_md) - len(cleaned)} chars = "
      f"{(len(raw_md) - len(cleaned)) / len(raw_md) * 100:.1f}%)")

Cleaning metrics:
  separators_removed: 0
  duplicate_headers_removed: 0
  footnote_sections_consolidated: 5
  unique_footnotes: 20

Cleaned length: 29096 chars (was 40776, removed 11680 chars = 28.6%)


In [10]:
# ── Flowchart extraction: extract_flowcharts() ─────────────────────────────

def extract_flowcharts(md_text: str) -> list[dict]:
    """
    Extract decision-tree flowcharts (Main Chart, Charts A–E) from
    the cleaned markdown into structured dicts suitable for YAML export.

    Each chart dict:
        {
            "chart_id": "Main" | "A" | "B" | "C" | "D" | "E",
            "title": "...",
            "reference_section": "4.3.2.2",
            "steps": [
                {"id": "A1", "question": "...", "yes": "...", "no": "..."},
                ...
            ]
        }
    """
    charts = []

    # ── Identify chart sections ────────────────────────────────────────────
    # Match headings like:
    #   ## Design changes ... – Main Chart
    #   ## Chart A
    #   ### Chart B - Change of the Design*
    #   # Chart E
    #   **Main Chart**
    chart_heading_pat = re.compile(
        r"^#{1,6}\s+.*?"
        r"(?:(Main\s+Chart)|(?:Chart\s+)([A-E]))"
        r"(?:\s*[:\-–—]\s*(.+?))?$",
        re.MULTILINE | re.IGNORECASE,
    )

    matches = list(chart_heading_pat.finditer(md_text))

    # Also capture bold-only labels like **Main Chart** that aren't headings
    body_pat = re.compile(
        r"(?:^|\n)\s*\*\*(?:(Main\s+Chart)|Chart\s+([A-E])[^*]*)\*\*",
        re.MULTILINE | re.IGNORECASE,
    )
    for bm in body_pat.finditer(md_text):
        # Only add if it's not already covered by a heading match within 200 chars
        near = any(abs(bm.start() - hm.start()) < 200 for hm in matches)
        if not near:
            matches.append(bm)

    # Sort by position in document
    matches.sort(key=lambda m: m.start())

    if not matches:
        print("⚠ No flowchart sections found in the markdown.")
        return charts

    for i, m in enumerate(matches):
        # Determine chart ID
        chart_letter = None
        for g in range(1, (m.lastindex or 0) + 1):
            val = m.group(g)
            if val:
                if "main" in val.lower():
                    chart_letter = "Main"
                else:
                    chart_letter = val.strip()
                break
        chart_letter = chart_letter or "Main"
        chart_title = ""
        if m.lastindex and m.lastindex >= 3 and m.group(3):
            chart_title = m.group(3).strip()

        # Get text block for this chart
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else min(start + 3000, len(md_text))
        block = md_text[start:end]

        # ── Parse decision steps ────────────────────────────────────────
        steps = []

        # Pattern: **Step B1**: question or **Step 1**: question or **Step A**: question
        step_re = re.compile(
            r"\*\*(?:Step\s+)?([A-Z]?\d+|[A-Z]|\d+)\*\*[:\s]*(.+?)(?=\n\s*[-*•]|\n\n|\Z)",
            re.DOTALL,
        )
        for sm in step_re.finditer(block):
            step_id = sm.group(1).strip()
            question = re.sub(r"\s+", " ", sm.group(2).strip()).rstrip("*").rstrip()
            if not question.endswith("?"):
                question += "?"

            after = block[sm.end():sm.end() + 500]
            yes_m = re.search(r"[-*•]\s*\*?\*?Yes\*?\*?\s*[→:–\-]\s*(.+?)(?:\n|$)", after, re.I)
            no_m = re.search(r"[-*•]\s*\*?\*?No\*?\*?\s*[→:–\-]\s*(.+?)(?:\n|$)", after, re.I)

            # If step_id is a single letter or single digit (Main Chart), prefix with "Main"
            if chart_letter == "Main" and len(step_id) == 1:
                sid = f"Main-{step_id}"
            elif step_id[0].isdigit() and chart_letter != "Main":
                sid = f"{chart_letter}{step_id}"
            else:
                sid = step_id

            step = {"id": sid, "question": question}
            if yes_m:
                step["yes"] = yes_m.group(1).strip()
            if no_m:
                step["no"] = no_m.group(1).strip()
            steps.append(step)

        # Fallback: bold question lines then yes/no
        if not steps:
            current_step = None
            idx = 0
            for line in block.split("\n"):
                s = line.strip()
                if not s:
                    continue
                qm = re.match(r"\*\*(.{10,}?)(\?)?\*\*", s)
                if qm:
                    idx += 1
                    q = qm.group(1).strip()
                    if not q.endswith("?"):
                        q += "?"
                    current_step = {"id": f"{chart_letter}{idx}", "question": q}
                    steps.append(current_step)
                    continue
                if current_step:
                    if re.match(r"[-*•]\s*\*?\*?Yes\b", s, re.I):
                        tail = re.sub(r"^[-*•]\s*\*?\*?Yes\*?\*?\s*[→:–\-]?\s*", "", s, flags=re.I)
                        current_step["yes"] = tail.strip() or "significant"
                    elif re.match(r"[-*•]\s*\*?\*?No\b", s, re.I):
                        tail = re.sub(r"^[-*•]\s*\*?\*?No\*?\*?\s*[→:–\-]?\s*", "", s, flags=re.I)
                        current_step["no"] = tail.strip() or "non-significant"

        ref_m = re.search(r"Section\s+(\d+(?:\.\d+)*)", block[:500])
        charts.append({
            "chart_id": chart_letter,
            "title": chart_title or ("Main Chart" if chart_letter == "Main" else f"Chart {chart_letter}"),
            "reference_section": ref_m.group(1) if ref_m else "",
            "steps": steps,
        })

    # ── Deduplicate: keep the entry with most steps per chart_id ──────
    deduped = {}
    for c in charts:
        cid = c["chart_id"]
        if cid not in deduped or len(c["steps"]) > len(deduped[cid]["steps"]):
            # Merge title from descriptive entry if current one is generic
            if cid in deduped and c["title"] == f"Chart {cid}" and deduped[cid]["title"] != f"Chart {cid}":
                c["title"] = deduped[cid]["title"]
            deduped[cid] = c
    charts = list(deduped.values())

    return charts


# ── Run flowchart extraction ───────────────────────────────────────────────
flowcharts = extract_flowcharts(cleaned)

print(f"Extracted {len(flowcharts)} flowchart(s):")
for fc in flowcharts:
    print(f"  Chart {fc['chart_id']}: \"{fc['title']}\" — {len(fc['steps'])} decision steps")
    for step in fc["steps"][:5]:
        yes_str = f" → Yes: {step.get('yes', '?')[:50]}" if 'yes' in step else ""
        no_str = f" → No: {step.get('no', '?')[:50]}" if 'no' in step else ""
        print(f"    {step['id']}: {step['question'][:70]}{yes_str}{no_str}")

if flowcharts:
    yaml_path = SANDBOX / "mdcg_2020_3_rev1_flowcharts.yaml"
    yaml_path.write_text(
        yaml.dump(flowcharts, default_flow_style=False, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print(f"\nFlowcharts YAML saved: {yaml_path}")

Extracted 6 flowchart(s):
  Chart C: "Software Change*" — 5 decision steps
    C1: New or major change of operating system or any component? → Yes: The change is considered significant per Art. 120( → No: Go to Step C2
    C2: New or modification of architecture or database structure, change of a → Yes: The change is considered significant per Art. 120( → No: Go to Step C3
    C3: Required user input replaced by closed loop algorithm? → Yes: The change is considered significant per Art. 120( → No: Go to Step C4
    C4: New user interface, medical feature, channel of interoperability or pr → Yes: The change is considered significant per Art. 120( → No: Go to Step C5
    C5: Does the change negatively affect the benefit/risk ratio of the device → Yes: The change is considered significant per Art. 120( → No: Return to Main Chart Question D
  Chart D: "Change of a substance or material*" — 4 decision steps
    D1: Change to or addition of a new material of human/animal origin? → Yes: The c

In [ ]:
# ── Assemble and save final clean output ────────────────────────────────────

output_path = SANDBOX / "mdcg_2020_3_rev1_clean.md"
output_path.write_text(cleaned, encoding="utf-8")
print(f"✓ Clean markdown saved: {output_path}")
print(f"  Size: {len(cleaned):,} chars")

# ── Validation checks ──────────────────────────────────────────────────────
title_count = len(re.findall(
    r"Medical Device Coordination Group Document",
    cleaned, re.IGNORECASE,
))
footnote_sections = len(re.findall(r"^#{1,3}\s*Footnotes?\s*$", cleaned, re.MULTILINE))

# Count detected section headings
section_headings = re.findall(r"^#{2,6}\s+\d+(?:\.\d+)*\s+", cleaned, re.MULTILINE)

print(f"\n── Validation ──")
print(f"  Title occurrences:     {title_count} (target: 1)")
print(f"  Footnote sections:     {footnote_sections} (target: 1)")
print(f"  Section headings:      {len(section_headings)}")
print(f"  Flowcharts extracted:  {len(flowcharts)}")

if title_count <= 2:
    print("  ✓ Header deduplication: PASS")
else:
    print(f"  ✗ Header deduplication: FAIL — {title_count} occurrences remain")

if footnote_sections <= 1:
    print("  ✓ Footnote consolidation: PASS")
else:
    print(f"  ✗ Footnote consolidation: FAIL — {footnote_sections} sections remain")

if len(flowcharts) >= 5:
    print("  ✓ Flowchart extraction: PASS")
else:
    print(f"  ⚠ Flowchart extraction: PARTIAL — expected ≥5, got {len(flowcharts)}")

In [ ]:
# ── Reusable wrapper: parse_mdcg_pdf() ──────────────────────────────────────

async def parse_mdcg_pdf(
    pdf_path: str | Path,
    output_dir: str | Path | None = None,
    api_key: str | None = None,
    tier: str = "agentic",
    custom_prompt: str | None = None,
) -> dict:
    """
    End-to-end MDCG PDF parsing pipeline.

    1. Upload & parse via LlamaParse v2
    2. Post-process (dedup headers, consolidate footnotes, fix hierarchy)
    3. Extract flowcharts into structured YAML
    4. Save outputs

    Returns:
        {
            "markdown": str,         # cleaned markdown
            "flowcharts": list,      # structured chart dicts
            "metrics": dict,         # cleaning metrics
            "output_files": dict,    # paths to saved files
        }
    """
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir) if output_dir else pdf_path.parent
    stem = pdf_path.stem
    key = api_key or os.environ.get("LLAMA_CLOUD_API_KEY", "")

    if not key:
        raise RuntimeError("No LLAMA_CLOUD_API_KEY provided")
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    prompt = custom_prompt or MDCG_CUSTOM_PROMPT

    # ── Step 1: Parse ──────────────────────────────────────────────────────
    client = AsyncLlamaCloud(api_key=key)

    print(f"Uploading {pdf_path.name}…")
    file_obj = await client.files.create(file=str(pdf_path), purpose="parse")

    print(f"Parsing with {tier} tier…")
    result = await client.parsing.parse(
        file_id=file_obj.id,
        tier=tier,
        version="latest",
        agentic_options={"custom_prompt": prompt},
        crop_box={"top": 0.07, "bottom": 0.04, "left": 0.0, "right": 0.0},
        output_options={"markdown": {"annotate_links": True}},
        expand=["markdown", "text"],
    )

    raw = "\n\n".join(p.markdown for p in result.markdown.pages)
    print(f"  Raw: {len(raw):,} chars across {len(result.markdown.pages)} pages")

    # ── Step 2: Clean ──────────────────────────────────────────────────────
    cleaned_md, cleaning_metrics = clean_mdcg_markdown(raw)
    print(f"  Cleaned: {len(cleaned_md):,} chars")

    # ── Step 3: Flowcharts ─────────────────────────────────────────────────
    charts = extract_flowcharts(cleaned_md)
    print(f"  Flowcharts: {len(charts)} extracted")

    # ── Step 4: Save ───────────────────────────────────────────────────────
    files = {}

    raw_path = output_dir / f"{stem}_raw.md"
    raw_path.write_text(raw, encoding="utf-8")
    files["raw_markdown"] = str(raw_path)

    clean_path = output_dir / f"{stem}_clean.md"
    clean_path.write_text(cleaned_md, encoding="utf-8")
    files["clean_markdown"] = str(clean_path)

    if charts:
        yaml_path = output_dir / f"{stem}_flowcharts.yaml"
        yaml_path.write_text(
            yaml.dump(charts, default_flow_style=False, allow_unicode=True, sort_keys=False),
            encoding="utf-8",
        )
        files["flowcharts_yaml"] = str(yaml_path)

    meta_path = output_dir / f"{stem}_meta.json"
    meta = {
        "source_pdf": pdf_path.name,
        "tier": tier,
        "pages": len(result.markdown.pages),
        "raw_chars": len(raw),
        "cleaned_chars": len(cleaned_md),
        "cleaning_metrics": cleaning_metrics,
        "flowcharts_count": len(charts),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    files["metadata"] = str(meta_path)

    print(f"  All files saved to {output_dir}/")
    return {
        "markdown": cleaned_md,
        "flowcharts": charts,
        "metrics": cleaning_metrics,
        "output_files": files,
    }

print("parse_mdcg_pdf() defined and ready to use.")

In [ ]:
# ── Test with second MDCG PDF ───────────────────────────────────────────────
# Uncomment and run to parse the second document:

# result2 = await parse_mdcg_pdf(
#     pdf_path=SANDBOX / "mdcg_2019_11_en.pdf",
#     output_dir=SANDBOX,
#     tier="agentic",
# )
# print(f"\nSecond PDF result: {result2['metrics']}")
# print(f"Flowcharts: {len(result2['flowcharts'])}")